# 08. Короткі запуски YOLO11s: рецепт проти перенавчання

У `04` криві val mAP50-95 (Ultralytics) досягають максимуму на 1-2 епосі (≈ 0.18) і до 25 епохи падають до ≈ 0.153, хоча train loss весь час спадає. `best.pt` це майже сирі COCO-ваги. Гіпотеза: у train лише 56 клипів зі скорельованих кадрів, тому довге навчання запам'ятовує сцени, а високий lr руйнує передтреновані ознаки.

Перевіряємо чотири рецепти по 6 епох на всьому `prepared/train` (stride 3), решта як у `04` (yolo11s, 960 px, seed 42, ті самі val-кадри, оцінка `common.evaluate` з ignore-зонами):
- `s_base`: стандартний рецепт (SGD, lr0 = 0.01, warmup 3 епохи), лише коротший;
- `s_lowlr`: lr0 = 0.002, косинусний розклад, warmup 1 епоха;
- `s_freeze10`: заморожені перші 10 шарів (backbone), решта як `s_base`;
- `s_aug`: сильніші аугментації (`mixup` 0.15, `scale` 0.7).

Оптимізатор заданий явно: при `optimizer=auto` і менш ніж 10000 ітерацій Ultralytics сам вибирає AdamW і ігнорує вказаний lr0.
Для кожного запуску рахуємо метрики для `best.pt` (за fitness Ultralytics) і `last.pt` окремо. Один запуск на конфігурацію, у val лише 7 клипів, тому різниці менші за ≈ 0.005 вважаємо шумом.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

import analysis as an
import common as c
import yolo_utils as yu
from ultralytics import YOLO

yu.setup()
c.set_seed()
P = c.paths()
OUT = P.out / "short_runs"
print("відновлено з Input:", c.restore_from_inputs(P, "short_runs"))
GT = P.prep / "val" / "annotations_eval.json"
BASE = dict(epochs=6, batch=16, workers=4, close_mosaic=2, optimizer="SGD", lr0=0.01, deterministic=False,
            plots=False, device=yu.gpu_devices())
if DEBUG:
    BASE.update(epochs=1, batch=4, workers=0, device="cpu")
DATA = yu.write_data_yaml(P.prep, OUT / "data_960.yaml", train_every=20 if DEBUG else 1)
print(BASE)

## 1. Експерименти

In [ ]:
EXPERIMENTS = {
    "s_base": dict(imgsz=960),
    "s_lowlr": dict(imgsz=960, lr0=0.002, cos_lr=True, warmup_epochs=1),
    "s_freeze10": dict(imgsz=960, freeze=10),
    "s_aug": dict(imgsz=960, mixup=0.15, scale=0.7),
}
RUN = list(EXPERIMENTS)
display(pd.DataFrame(EXPERIMENTS).T.fillna("-"))


def evaluate_weights(weights, imgsz, per_class=False):
    dets, _ = yu.predict_prepared(YOLO(weights), P.prep, "val", imgsz=imgsz)
    return dets, c.evaluate(GT, dets, per_class=per_class)


def run_experiment(name):
    args = {**BASE, **EXPERIMENTS[name]}
    if DEBUG:
        args["imgsz"] = 320
    run = yu.train(None, "yolo11s.pt", DATA, OUT, name, **args)
    dets, metrics = evaluate_weights(run / "weights" / "best.pt", args["imgsz"], per_class=True)
    c.save_json(dets, OUT / name / "val_dets.json")
    last = evaluate_weights(run / "weights" / "last.pt", args["imgsz"])[1]
    log = yu.results_csv(run)
    fitness = 0.1 * log["metrics/mAP50(B)"] + 0.9 * log["metrics/mAP50-95(B)"]
    hours = float(log["time"].iloc[-1]) / 3600
    ngpu = len(args["device"]) if isinstance(args["device"], list) else 1
    return an.log_result(P.results, name, "short-run recipe", metrics, epochs=args["epochs"],
                         best_epoch=int(log["epoch"][fitness.idxmax()]), mAP_last=last["mAP"],
                         wall_hours=hours, gpu_hours=hours * ngpu, imgsz=args["imgsz"],
                         config=str(EXPERIMENTS[name]))


for name in RUN:
    print("=" * 20, name)
    run_experiment(name)
    torch.cuda.empty_cache()

## 2. Результати

Основна таблиця: `common.evaluate` на val з ignore-зонами. Нижче для довідки результати 25-епохових запусків з `04` і `yolo_s6` з `05` (з'являються, якщо їхні виходи підключені як Input).

In [ ]:
COLS = ["mAP", "mAP_last", "best_epoch", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "gpu_hours"]
res = an.all_results(P).set_index("name")
mine = res.loc[[n for n in RUN if n in res.index]]
display(mine[[col for col in COLS if col in mine]].round(3))
ref = res.loc[[n for n in ("yolo_s6", "yolo_base", "yolo_p2", "yolo_tal13") if n in res.index]]
display(ref[["mAP", "AP_small", "AP_medium", "AP_large", "gpu_hours"]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for name in RUN:
    if not (OUT / name / "results.csv").exists():
        continue
    r = yu.results_csv(OUT / name)
    axes[0].plot(r.epoch, r["train/box_loss"], marker=".", label=name)
    axes[1].plot(r.epoch, r["metrics/mAP50-95(B)"], marker=".", label=name)
    axes[2].plot(r.epoch, r["metrics/recall(B)"], marker=".", label=name)
for ax, t in zip(axes, ["box loss (train)", "val mAP50-95 (Ultralytics, без ignore)", "val recall (Ultralytics)"]):
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("епоха")
axes[1].legend(fontsize=8)
fig.savefig(P.images / "short_runs_curves.png", bbox_inches="tight", dpi=130)
plt.show()

## Висновки

<!-- Заповнюється після запуску з реальних таблиць: чи підтвердилась гіпотеза про перенавчання (best_epoch і різниця mAP та mAP_last), який рецепт дав найкращий mAP і AP_small порівняно з s_base і з 25-епоховими запусками з 04, скільки це коштувало GPU-годин, який рецепт беремо для більших моделей і сідів. -->